Script Overview:
1. Filters the URLs and downloads them through retrieval of the sitemap
2. Scraping HTML content from 2018 to 2024

1. Filters the URLs and downloads them

In [11]:
import requests
import xmltodict
import os
from datetime import datetime

In [9]:
url = "https://www.cbs.nl/nl-nl/sitemaps/sitemap"
response = requests.get(url)

In [12]:
if response.status_code == 200:
    current_date = datetime.now().strftime("%d_%m_%Y")
    # create a file path 
    file_path = f"sitemap/sitemap_{current_date}.xml"
    # save the sitemap to the file
    with open(file_path, "wb") as file:
        file.write(response.content)

    print(f"Sitemap data saved to {file_path}")
else:
    print(f"Failed to retrieve this sitemap {url} (status code: {response.status_code})")
    exit(1)

Sitemap data saved to sitemap/sitemap_25_11_2024.xml


2. Scraping HTML content from 2018 to 2024

In [ ]:

site_map_file_path = "sitemap//sitemap_25_11_2024.xml"

with open(site_map_file_path, 'r') as file:
    sitemap = xmltodict.parse(file.read())

urls = sitemap["urlset"]["url"]

# filtering dates and articles

start_date = datetime(2018,1,1)
end_date = datetime(2024, 11, 25)

filtered_urls = [
    url["loc"] for url in urls 
    if "https://www.cbs.nl/nl-nl/nieuws/" in url["loc"]
    and start_date <= datetime.strptime(url["lastmod"], "%Y-%m-%dt%H:%M:%SZ") <= end_date
    ]

In [56]:
# making the output directory
output_dir = "downloaded_html_files"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:

"""
This script downloads HTML content from a list of previously filtered URLs.

For each URL:
- It constructs a filename based on the last part of the URL.
- It skips downloading if the  file already exists.
- It sends a GET request and saves the HTML content to a local file if successful.
- If an error occurs or the request fails, the URL is logged for review.

At the end, all URLs that failed to download are printed.
"""

error_URLS = []

for filtered_url in filtered_urls:
    try:
        # create a valid filename from the URL 
        file_name = filtered_url.split("/")[-1] + ".html"
        file_path = os.path.join(output_dir, file_name)
        
        # check it already exists 
        if os.path.exists(file_path):
            print(f"File {file_name} already exists. Skipping Download")
            continue
        
        
        
        page_response = requests.get(filtered_url)

        if page_response.status_code == 200:
            page_content = page_response.text
            
            # save the HTML content to a file

            with open(file_path, "w", encoding = "utf-8") as file:
                file.write(page_content)

            print(f"Saved HTML for {filtered_url} to {file_path}")
        
        else:
            print(f"Failed to retreive {filtered_url}. Status code: {page_response.status_code}")
            error_URLS.append(filtered_urls)
    
    except Exception as e:
        print(f"Error downloading {filtered_url}: {e}")
        error_URLS.append(filtered_url)
        continue


if error_URLS:
    print("\nError URLS:")
    for url in error_URLS:
        print(url)